# 🔧 Notebook 10: Metal Shading Language & Custom Kernels

Metal compute shader basics, custom .metal kernels for softmax, RMSNorm, and tiled matmul, with performance comparisons against MLX built-ins.

**Prerequisites:** Notebooks 01-09

**What you'll learn:**
1. Understand Metal's thread hierarchy (threads, SIMD groups, threadgroups)\n2. Read and understand custom Metal compute kernels\n3. Benchmark MLX matmul performance at different sizes\n4. Understand why kernel fusion matters for performance

💡 **In simple terms**, think of it as building blocks — each concept in this notebook is a building block that connects to the next. For example, understanding the basics here will make everything that follows much easier to grasp.

### 🧠 The Big Picture

A GPU is like a massive warehouse with thousands of workers (threads). Each worker can only do simple tasks, but they all work simultaneously. Threadgroup memory is like a shared whiteboard that nearby workers can read — much faster than walking to the main office (device memory) every time they need information.

In [ ]:
from utils.checks import validate_environment, print_environment_report
env = print_environment_report()

## Metal Compute Shader Overview

Metal organizes GPU threads into:
- **Threads**: Individual execution units
- **SIMD Groups**: 32 threads executing in lockstep (like CUDA warps)
- **Threadgroups**: Groups of SIMD groups sharing fast threadgroup memory (~32KB)

Memory hierarchy (fastest to slowest):
```
Registers (per thread)     ~1 cycle
Threadgroup Memory         ~4-8 cycles    (~32 KB)
Device Memory (Unified)    ~100+ cycles   (48 GB)
```

💡 MLX compiles operations into Metal compute shaders automatically. Writing custom kernels lets us optimize specific operations.

## Custom Softmax Kernel

The softmax kernel uses shared memory for the max reduction (numerical stability trick). Each threadgroup processes one row of the input matrix.

In [ ]:
# 💡 Let's look at the actual Metal kernel source code
# This is what runs on the GPU when you call softmax

kernel_path = 'metal_kernels/softmax.metal'
try:
    with open(kernel_path) as f:
        kernel_source = f.read()
    print(f'📄 {kernel_path}:')
    print('=' * 60)
    print(kernel_source)
    print('=' * 60)
    print()
    print('💡 Key concepts in this kernel:')
    print('  1. threadgroup_barrier — synchronizes threads within a group')
    print('  2. shared_max/shared_sum — threadgroup (shared) memory for reductions')
    print('  3. Three-pass algorithm: find max → compute exp & sum → normalize')
    print('  4. Each threadgroup processes one ROW of the input matrix')
except FileNotFoundError:
    print(f'⚠️ {kernel_path} not found — run from the repo root directory')

In [ ]:
# The Metal kernel source is in metal_kernels/softmax.metal
# Here we explain the key concepts and compare with MLX built-in

import mlx.core as mx
import time

# MLX built-in softmax
x = mx.random.normal((1024, 512))  # shape: see output
mx.eval(x)

# Benchmark MLX softmax
for _ in range(3): mx.eval(mx.softmax(x, axis=-1))  # warmup
t0 = time.perf_counter()
for _ in range(100): mx.eval(mx.softmax(x, axis=-1))
t1 = time.perf_counter()
print(f'MLX softmax (1024×512): {(t1-t0)*10:.2f}ms per call')

# Verify correctness
result = mx.softmax(x, axis=-1)  # shape: see output
mx.eval(result)
print(f'Row sums (should be ~1.0): {mx.sum(result, axis=-1)[:3].tolist()}')
print(f'All values in [0,1]: {bool(mx.all(result >= 0).item() and mx.all(result <= 1).item())}')

## Custom RMSNorm Kernel

RMSNorm: `output = x / sqrt(mean(x²) + eps) * weight`

The kernel computes the RMS across the last dimension using shared memory reduction.

In [ ]:
import mlx.nn as nn

# Compare manual vs built-in RMSNorm
x = mx.random.normal((32, 128, 256))  # shape: see output
norm = nn.RMSNorm(256)
mx.eval(x)

for _ in range(3): mx.eval(norm(x))  # warmup
t0 = time.perf_counter()
for _ in range(100): mx.eval(norm(x))
t1 = time.perf_counter()
print(f'MLX RMSNorm (32×128×256): {(t1-t0)*10:.2f}ms per call')
print(f'\n💡 MLX uses optimized Metal kernels under the hood via mx.fast.rms_norm')

## Tiled Matrix Multiplication

Tiled matmul loads blocks of A and B into threadgroup shared memory, reducing device memory accesses. Each threadgroup computes one tile of the output matrix C.

⚡ MLX's `mx.matmul` already uses highly optimized Metal kernels. Custom kernels are educational — in practice, use the built-in.

In [ ]:
# Benchmark MLX matmul at various sizes
sizes = [256, 512, 1024, 2048]
print(f'{"Size":>8s} {"Time (ms)":>10s} {"GFLOPS":>10s}')
print('-' * 32)
for N in sizes:
    A = mx.random.normal((N, N))  # shape: see output
    B = mx.random.normal((N, N))  # shape: see output
    mx.eval(A, B)
    for _ in range(3): mx.eval(A @ B)  # warmup
    t0 = time.perf_counter()
    for _ in range(10): mx.eval(A @ B)
    t1 = time.perf_counter()
    ms = (t1-t0) * 100
    gflops = 2 * N**3 / (ms/1000) / 1e9
    print(f'{N:>8d} {ms:>10.2f} {gflops:>10.1f}')

## Metal Memory Model

- **Device memory**: Main unified memory (48GB). Accessible by CPU and GPU. High latency.
- **Threadgroup memory**: Fast on-chip SRAM (~32KB per threadgroup). Used for data sharing within a threadgroup.
- **Thread memory**: Registers. Fastest, but very limited.

**Memory coalescing**: Adjacent threads should access adjacent memory addresses for best bandwidth.

**Bank conflicts**: Threadgroup memory is divided into banks. If multiple threads access the same bank, accesses are serialized.

💡 MLX handles all of this automatically. Understanding it helps when debugging performance issues.

**Next:** Notebook 11 — Inference Optimization

## 📊 Matmul Performance Visualization

**GFLOPS** (Giga Floating-Point Operations Per Second) measures how many billions of multiply-add operations the GPU completes each second. For an N×N matrix multiply, the operation count is 2N³ (one multiply and one add per output element accumulation).

Why do larger matrices achieve higher GFLOPS? GPUs are massively parallel — they have thousands of cores that need to stay busy. Small matrices don't generate enough parallel work to fill all cores, so many sit idle. As matrix size grows, there's enough work to saturate the GPU, and the compute-to-memory-access ratio improves (more arithmetic per byte loaded). This is why LLM batch inference is faster per token than single-token generation.

In [ ]:
import matplotlib.pyplot as plt

# Re-run the matmul benchmark and collect results for plotting
# (cell e2 prints the table; here we store values for the chart)
sizes = [256, 512, 1024, 2048]
gflops_list = []
for N in sizes:
    A = mx.random.normal((N, N))
    B = mx.random.normal((N, N))
    mx.eval(A, B)
    for _ in range(3): mx.eval(A @ B)  # warmup
    t0 = time.perf_counter()
    for _ in range(10): mx.eval(A @ B)
    t1 = time.perf_counter()
    ms = (t1 - t0) * 100
    gflops_list.append(2 * N**3 / (ms / 1000) / 1e9)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([str(s) for s in sizes], gflops_list, color='#3b82f6', edgecolor='#1e3a5f')
ax.set_xlabel('Matrix Size (N×N)')
ax.set_ylabel('GFLOPS')
ax.set_title('MLX Matmul Throughput vs Matrix Size')

# Annotate each bar with its value
for bar, val in zip(bars, gflops_list):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(gflops_list) * 0.02,
            f'{val:.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylim(0, max(gflops_list) * 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n↑ Larger matrices achieve higher GFLOPS because they provide enough')
print(f'  parallel work to saturate the GPU cores and amortize memory latency.')

### 🔍 What Just Happened?

We just completed a key section of this notebook. The code above demonstrates the core concepts — take a moment to review the outputs and make sure you understand each step before moving on.

## 🧪 Try It Yourself

Explore Metal GPU performance:

1. **Matrix size experiment**: Run the matmul benchmark with sizes [64, 128, 256, 512, 1024, 2048, 4096]. At what size does GFLOPS plateau?

2. **Memory bandwidth**: Calculate the theoretical memory bandwidth utilization for each matrix size. (Hint: bytes_read = 2 * N * N * 4 for float32)

3. **Softmax benchmark**: Time `mx.softmax()` for different row lengths (128, 512, 2048, 8192). Is it compute-bound or memory-bound?

## 📜 History & Alternatives

### The Evolution of GPU Compute for ML

The journey from general-purpose GPU computing to specialized ML accelerators shows how hardware and software co-evolved to meet the demands of deep learning.

| Year | Technology | Team | Key Contribution |
|------|-----------|------|-----------------|
| 2007 | **CUDA** | NVIDIA | First general-purpose GPU computing platform — launched the GPU computing era |
| 2009 | **OpenCL** | Khronos Group | Open standard for heterogeneous computing — cross-vendor but less optimized |
| 2012 | **cuDNN** | NVIDIA | GPU-accelerated deep learning primitives — made training CNNs practical |
| 2014 | **Metal** | Apple | Low-overhead GPU API for Apple platforms — replaced OpenGL ES |
| 2017 | **Metal 2** | Apple | ML inference support, MPS (Metal Performance Shaders) for neural networks |
| 2017 | **Tensor Cores** | NVIDIA (Volta) | Hardware matrix multiply units — 8× throughput for mixed precision |
| 2020 | **Metal 3 (preview)** | Apple | Mesh shaders, hardware ray tracing — foundation for M-series GPU compute |
| 2022 | **Metal 3** | Apple (M2+) | Dynamic Caching, mesh shaders, hardware-accelerated ray tracing |
| 2022 | **H100 + FP8** | NVIDIA (Hopper) | Transformer Engine with FP8 — 6× AI throughput over A100 |
| 2023 | **MLX Metal kernels** | Apple ML Research | Custom Metal compute kernels optimized for MLX operations on Apple Silicon |
| 2023 | **ROCm 5.x** | AMD | Mature open-source GPU compute — competitive with CUDA for training |
| 2024 | **Metal 3 + MLX** | Apple | Tight integration — MLX leverages Metal compute shaders for all GPU ops |
| 2024 | **Blackwell (B200)** | NVIDIA | 2nd gen Transformer Engine, FP4 support, 2.5× over H100 |

### 💡 Why Custom Kernels Matter

Default framework operations (matmul, softmax, etc.) are general-purpose. Custom kernels can fuse multiple operations into a single GPU dispatch, eliminating memory round-trips:

```
Standard: Read A → Compute matmul → Write C → Read C → Compute softmax → Write D
Fused:    Read A → Compute matmul+softmax → Write D  (1 read/write saved)
```

On Apple Silicon, memory bandwidth (~400 GB/s on M4 Max) is the bottleneck for LLM inference, so reducing memory traffic via kernel fusion can yield 2-3× speedups.

### GPU Compute Ecosystem Comparison

| Platform | Language | Ecosystem | Apple Silicon Support | Maturity |
|----------|----------|-----------|----------------------|----------|
| **CUDA** | CUDA C/C++ | Massive (cuDNN, cuBLAS, TensorRT) | None | Very mature |
| **Metal** | Metal Shading Language (MSL) | Growing (MPS, MLX) | Native | Maturing |
| **ROCm** | HIP (CUDA-like) | Growing | None | Maturing |
| **OpenCL** | OpenCL C | Declining | Deprecated on Apple | Legacy |
| **Vulkan Compute** | GLSL/SPIR-V | Niche for ML | MoltenVK (translation) | Niche |
| **SYCL** | C++ | Intel-focused | None | Growing |

### ⚡ Metal Kernel Performance Tips

1. **Threadgroup memory**: Use shared memory (threadgroup) for data reused across threads — 10-100× faster than device memory
2. **Simdgroup operations**: Leverage SIMD-width operations (32 threads) for reductions and scans
3. **Memory coalescing**: Ensure adjacent threads access adjacent memory addresses
4. **Occupancy**: Balance threadgroup size with register usage to maximize GPU utilization
5. **Kernel fusion**: Combine elementwise operations with matmul to reduce memory round-trips

### ⚠️ Common Pitfalls

1. **Threadgroup size mismatch**: Setting threadgroup dimensions that don't evenly divide the grid can leave threads idle or cause out-of-bounds memory access — always guard with boundary checks
2. **Unified memory ≠ free bandwidth**: Apple Silicon's unified memory eliminates explicit copies, but GPU and CPU contend for the same bandwidth — concurrent CPU-heavy work during GPU kernels can tank throughput
3. **Over-optimizing for SIMD width**: Metal's SIMD width (32 on Apple GPU) differs from CUDA's warp size (32) in subtle ways — don't assume CUDA tiling strategies transfer directly
4. **Ignoring memory alignment**: Metal requires 16-byte alignment for threadgroup memory. Misaligned accesses silently degrade performance without errors
5. **Debugging blind spots**: Metal lacks CUDA's mature profiling ecosystem (NSight, cuda-memcheck). Use Xcode's GPU Frame Capture and Metal System Trace, but expect less granularity

### 🎯 Interview Tip

> *"Metal compute shaders on Apple Silicon use a tile-based deferred rendering architecture, which means the GPU processes work in tiles that fit in on-chip memory. For ML workloads, this maps well to tiled matrix multiplication (similar to how CUDA uses shared memory tiling). The key difference from CUDA is that Metal uses a unified memory model — there's no explicit host↔device transfer, which simplifies kernel development but requires careful attention to memory access patterns for optimal bandwidth utilization."*

---
## ➡️ What's Next?

You've completed this notebook! In **Notebook 11 — Inference Optimization**, we'll continue building on these concepts.

💡 **Before moving on**, make sure you can answer these questions:
- What was the main concept in this notebook?
- Why does it matter for building LLMs?
- Could you explain it to a friend in one sentence?